# Stage 4 — Cityscapes Semantic Segmentation on Video

This notebook runs a pretrained **SegFormer-B5** (84.7M params, 1024×1024 input) on the original dashcam video and produces a pixel-level semantic segmentation overlay.

**Model:** `nvidia/segformer-b5-finetuned-cityscapes-1024-1024` via Hugging Face Transformers.

**Output:** `runs_output/segmentation/cityscapes_segmented.mp4`

**Note:** SegFormer-B5 is a hybrid architecture (Vision Transformer encoder + CNN/MLP decoder). It produces high-quality masks but is **not real-time**. This is a quality demonstration — real-time detection is already covered by Stage 1/2b (YOLO).

**Quick-test gate:** Set `PROCESS_SECS = 10` to validate the pipeline on a 10-second clip before a full render.

In [ ]:
!pip install -q transformers

In [ ]:
import os
import cv2
import numpy as np
import torch
import torch.nn.functional as F
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from transformers import AutoImageProcessor, AutoModelForSemanticSegmentation

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

## Load pretrained SegFormer-B5 (Cityscapes)

SegFormer-B5 is a hybrid architecture: a hierarchical Vision Transformer encoder captures global context, while a lightweight MLP decoder paints sharp pixel boundaries.

In [ ]:
MODEL_ID = "nvidia/segformer-b5-finetuned-cityscapes-1024-1024"

image_processor = AutoImageProcessor.from_pretrained(MODEL_ID)
seg_model = AutoModelForSemanticSegmentation.from_pretrained(MODEL_ID).to(DEVICE)
seg_model.eval()

print("SegFormer-B5 loaded on", DEVICE)

## Cityscapes 19-class label palette

Official Cityscapes color map. The SegFormer checkpoint is trained on **19 train IDs** (0–18). The palette reserves 30 entries for safety, but the model only predicts indices 0–18 — indices 19–29 are void/unlabeled.

In [ ]:
CITYSCAPES_LABELS = [
    "road", "sidewalk", "building", "wall", "fence", "pole",
    "traffic light", "traffic sign", "vegetation", "terrain", "sky",
    "person", "rider", "car", "truck", "bus", "train",
    "motorcycle", "bicycle",
    "void"  # remaining indices map to void/unlabeled
]

# Extend to 30 labels with void for indices 19-29
CITYSCAPES_LABELS = CITYSCAPES_LABELS + ["void"] * (30 - len(CITYSCAPES_LABELS))

# Official Cityscapes color palette (BGR order for OpenCV)
CITYSCAPES_COLORS = np.array([
    [128,  64, 128],  # 0  road
    [244,  35, 232],  # 1  sidewalk
    [ 70,  70,  70],  # 2  building
    [102, 102, 156],  # 3  wall
    [190, 153, 153],  # 4  fence
    [153, 153, 153],  # 5  pole
    [250, 170,  30],  # 6  traffic light
    [220, 220,   0],  # 7  traffic sign
    [107, 142,  35],  # 8  vegetation
    [152, 251, 152],  # 9  terrain
    [ 70, 130, 180],  # 10 sky
    [220,  20,  60],  # 11 person
    [255,   0,   0],  # 12 rider
    [  0,   0, 142],  # 13 car
    [  0,   0,  70],  # 14 truck
    [  0,  60, 100],  # 15 bus
    [  0,  80, 100],  # 16 train
    [  0,   0, 230],  # 17 motorcycle
    [119,  11,  32],  # 18 bicycle
    [  0,   0,   0],  # 19 void
    [  0,   0,   0],  # 20 void
    [  0,   0,   0],  # 21 void
    [  0,   0,   0],  # 22 void
    [  0,   0,   0],  # 23 void
    [  0,   0,   0],  # 24 void
    [  0,   0,   0],  # 25 void
    [  0,   0,   0],  # 26 void
    [  0,   0,   0],  # 27 void
    [  0,   0,   0],  # 28 void
    [  0,   0,   0],  # 29 void
], dtype=np.uint8)

print(f"Palette loaded: {len(CITYSCAPES_COLORS)} classes")

## Process video frame-by-frame

Set `PROCESS_SECS` to limit output for quick validation. `None` = process full video.

In [ ]:
# ------------------------------------------------------------------
# Configuration
# ------------------------------------------------------------------
VIDEO_INPUT = "original_videos/dashcam.mp4"
OUTPUT_DIR = Path("runs_output/segmentation")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_VIDEO = OUTPUT_DIR / "cityscapes_segmented.mp4"

# Set to e.g. 10 for quick validation; None for full video
PROCESS_SECS = None
ALPHA = 0.5  # segmentation overlay blend

# ------------------------------------------------------------------
# Open input
# ------------------------------------------------------------------
cap = cv2.VideoCapture(VIDEO_INPUT)
assert cap.isOpened(), f"Could not open video: {VIDEO_INPUT}"

fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

MAX_FRAMES = int(fps * PROCESS_SECS) if PROCESS_SECS else frame_count
MAX_FRAMES = min(MAX_FRAMES, frame_count)

print(f"FPS: {fps}")
print(f"Resolution: {width}x{height}")
print(f"Total frames: {frame_count}")
print(f"Will process: {MAX_FRAMES} frames ({MAX_FRAMES / fps:.1f}s)")

# ------------------------------------------------------------------
# Video writer
# ------------------------------------------------------------------
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(str(OUTPUT_VIDEO), fourcc, float(fps), (width, height))
if not writer.isOpened():
    cap.release()
    writer.release()
    raise RuntimeError(
        f"cv2.VideoWriter failed to open with fourcc 'mp4v' for {OUTPUT_VIDEO}. "
        "OpenCV build is missing the required codec."
    )

# ------------------------------------------------------------------
# Frame processing loop
# ------------------------------------------------------------------
completed = False
try:
    for i in tqdm(range(MAX_FRAMES)):
        ret, frame = cap.read()
        if not ret:
            break

        # ---- Prepare for SegFormer ----
        pil = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        inputs = image_processor(images=pil, return_tensors="pt")
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

        # ---- Inference ----
        # logits shape: (1, num_labels, H, W) — typically 19 for Cityscapes train IDs
        with torch.no_grad():
            outputs = seg_model(**inputs)
            logits = outputs.logits

        # ---- Upsample to original resolution ----
        # CRITICAL: use nearest-neighbor to preserve sharp class boundaries
        upsampled = F.interpolate(
            logits,
            size=(height, width),
            mode="nearest",
        )
        seg_map = upsampled.argmax(dim=1).squeeze(0).cpu().numpy()  # (H, W)

        # ---- Colorize ----
        mask_color = CITYSCAPES_COLORS[seg_map]  # (H, W, 3)

        # ---- Alpha blend ----
        blended = cv2.addWeighted(frame, 1 - ALPHA, mask_color, ALPHA, 0)
        writer.write(blended)

    completed = True
finally:
    cap.release()
    writer.release()
    if not completed and OUTPUT_VIDEO.is_file():
        try:
            OUTPUT_VIDEO.unlink()
            print(f"Removed partial output: {OUTPUT_VIDEO}")
        except OSError as e:
            print(f"Warning: could not remove partial output: {e}")

print(f"\nSaved segmentation video to:")
print(OUTPUT_VIDEO)

## Inspect one sample frame

Display original, color mask, and blended side-by-side for visual verification.

In [ ]:
import matplotlib.pyplot as plt

SAMPLE_SEC = 5.0  # pick a frame at this timestamp

cap = cv2.VideoCapture(VIDEO_INPUT)
cap.set(cv2.CAP_PROP_POS_MSEC, SAMPLE_SEC * 1000)
ret, frame = cap.read()
cap.release()

if not ret:
    raise RuntimeError("Could not read sample frame")

# Run segmentation on this single frame
pil = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
inputs = image_processor(images=pil, return_tensors="pt")
inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

with torch.no_grad():
    outputs = seg_model(**inputs)
    logits = outputs.logits

h, w = frame.shape[:2]
upsampled = F.interpolate(logits, size=(h, w), mode="nearest")
seg_map = upsampled.argmax(dim=1).squeeze(0).cpu().numpy()
mask_color = CITYSCAPES_COLORS[seg_map]
blended = cv2.addWeighted(frame, 1 - ALPHA, mask_color, ALPHA, 0)

# Plot
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
axes[0].set_title("Original")
axes[0].axis("off")

axes[1].imshow(mask_color[..., ::-1])  # BGR → RGB for matplotlib
axes[1].set_title("Segmentation Mask")
axes[1].axis("off")

axes[2].imshow(cv2.cvtColor(blended, cv2.COLOR_BGR2RGB))
axes[2].set_title(f"Blended (alpha={ALPHA})")
axes[2].axis("off")

plt.tight_layout()
plt.show()

## Next Steps

- If the segmentation quality looks good on the sample frame, re-run the video cell with `PROCESS_SECS = None` for the full video.
- For the final report, compare one frame from Stage 2b (YOLO boxes) with Stage 4 (pixel masks) side-by-side to illustrate detection vs. segmentation trade-offs.